# M3 — Data preprocessing and integrity validation
Synthetic ER, BA, SBM, directed ER. Real SNAP preprocessing is implemented via the M2 loader; actual download/inference is pending.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'm34.py').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from m34 import normalize, encode, decode, pairs, build_records, pilot_graphs, FORMATS


## Preserve direction and isolates; remove duplicate arcs and self-loops

In [ ]:
g = nx.DiGraph()
g.add_edges_from([(10,20),(10,20),(20,20),(20,30)])
g.add_node(40)
h = normalize(g)
assert h.is_directed() and len(h)==4 and h.number_of_edges()==2 and nx.number_of_selfloops(h)==0
assert not h.has_edge(1,0) and 3 in h
print('Fixture passed:', pairs(h))


## Independent graphs, matched balanced queries and round-trip checks

In [ ]:
cost, questions = build_records(pilot_graphs())
assert not set(questions[questions.split=='train'].graph_id) & set(questions[questions.split=='test'].graph_id)
assert cost.roundtrip.all()
print('Graph representations:',len(cost),'Unique graphs:',cost.graph_id.nunique())
print('Unique questions:', questions.question_id.nunique())
display(questions.groupby(['split','encoding','answer']).size().unstack(fill_value=0))


In [ ]:
qid = questions.question_id.iloc[0]
for row in questions[questions.question_id==qid].itertuples():
    print(row.encoding, '\n', row.prompt[:350], '\nReference:', row.answer, '\n')


## Representation cost (not LLM tokenizer tokens)

In [ ]:
display(cost.groupby('encoding').agg(graphs=('graph_id','size'),mean_bytes=('bytes','mean'),lossless=('roundtrip','mean')).round(3))
cost.groupby('encoding').bytes.mean().sort_values().plot.barh(title='Mean graph encoding size')
plt.xlabel('UTF-8 bytes'); plt.tight_layout(); plt.show()


## Real SNAP pipeline
Run python m34.py --mode llm --snap --limit 24 from the repository root after installing optional LLM dependencies. This uses the same normalization, paired encodings and exact labels with the M2 loader. No real SNAP predictions are claimed here.